<a href="https://colab.research.google.com/github/klaraseli/VMM-projekt/blob/misus/testovani_hypotez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Korekce na vícenásobné testování (Bonferroni)

**Co děláme:** Testujeme 7 hypotéz na stejných datech. Proto hladinu významnosti
neponecháváme na 0,05, ale upravujeme ji na **α = 0,05 / 7 ≈ 0,007**.

**Proč:** Každý test s α = 0,05 má 5% šanci, že vyjde „významně“ čistě náhodou,
i když ve skutečnosti žádný efekt neexistuje. Při 7 testech je šance, že aspoň
jeden z nich vyjde významně omylem, přibližně 1 − 0,95⁷ ≈ 30 %. Bonferroniho
korekce toto riziko drží na ~5 % pro celou sadu hypotéz.

**Jak to čteme:** Hypotézu považujeme za potvrzenou, jen když platí p < 0,007.
Ekvivalentně můžeme p-hodnotu vynásobit 7 a porovnat ji s 0,05.

**Cena:** Korekce je konzervativní. Slabé, ale skutečné efekty tak mohou
vyjít jako nevýznamné.

## Další rozhodnutí v analýze

- **Jednostranné testy** používáme tam, kde je směr hypotézy daný předem
  (H1, H3, H4, H5, H6). U H2 a H7 používáme oboustranné testy.
- **Ordinální proměnné** (např. `studytime`, `max_edu`) testujeme neparametricky
  (Spearman, Mannův–Whitneyův test), protože t-test předpokládá spojitá data.
- **H7 (město × vesnice)** testujeme po zohlednění školy (`G3 ~ address + school`),
  protože školy se liší jak průměrnou známkou, tak podílem městských studentů.
- **Velikost účinku** (ρ, Cohenovo d) uvádíme vedle p-hodnoty, protože při
  n = 649 vychází významně i velmi malé rozdíly.

In [23]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import scipy.stats as stats
import numpy as np
import statsmodels.formula.api as smf

# Načtení dat (spustí se jen jednou)
file_path = "/content/drive/MyDrive/Colab Notebooks/student_performance_clean_final.csv"
df = pd.read_csv(file_path)

# Příprava agregovaných sloupců pro všechny budoucí výpočty
df['max_edu'] = df[['Medu', 'Fedu']].max(axis=1)
df['alkohol_celkem'] = df['Dalc'] + df['Walc']

# Bonferroniho korekce pro 7 hypotéz
ALPHA = 0.05 / 7

# Jednotná funkce pro vyhodnocení a hezký výpis s českými desetinnými čárkami
def zaver(p):
    p_text = "< 0,001" if p < 0.001 else f"{p:.4f}".replace(".", ",")
    rozhodnuti = "zamítáme H0" if p < ALPHA else "nezamítáme H0"
    alpha_text = f"{ALPHA:.4f}".replace(".", ",")
    print(f"P-hodnota: {p_text}")
    print(f"ROZHODNUTÍ: {rozhodnuti} (α = {alpha_text})")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# === H1: Samostudium vs. Známka G3 ===
r_stat, p_hodnota = stats.spearmanr(df['studytime'], df['G3'], alternative='greater')
print("=== H1: Samostudium vs. Známka (G3) ===")
print(f"Spearmanovo rho (r): {r_stat:.3f}".replace(".", ","))
zaver(p_hodnota)



=== H1: Samostudium vs. Známka (G3) ===
Spearmanovo rho (r): 0,275
P-hodnota: < 0,001
ROZHODNUTÍ: zamítáme H0 (α = 0,0071)


**Proč tento test:** `studytime` má jen čtyři uspořádané hodnoty (1–4) a `G3` není normálně rozdělené, takže Pearsonova korelace není vhodná. Spearmanův koeficient pracuje s pořadími a zachytí jakýkoli monotónní trend. Hypotéza má předem daný směr (více samostudia → vyšší skóre), proto používáme jednostranný test.

**Co jsme zjistili:** ρ = 0,275, p < 0,001, tedy pod Bonferroniho hranicí 0,007, a H0 zamítáme. Studenti, kteří uvádějí více samostudia, mají v průměru vyšší `G3`. Vztah je slabý až mírný: samostudium vysvětluje zhruba 8 % variability pořadí známek (ρ² ≈ 0,08). Jde o souvislost, ne o důkaz příčiny.

In [24]:
zadani = df[df['romantic'] == 1]['studytime']
nezadani = df[df['romantic'] == 0]['studytime']
u_stat, p_hodnota = stats.mannwhitneyu(zadani, nezadani, alternative='two-sided')
r_rb = (2 * u_stat) / (len(zadani) * len(nezadani)) - 1

print("=== H2: Romantický vztah vs. Čas na studium ===")
print(f"Průměr studytime - zadaní: {zadani.mean():.2f}".replace(".", ","))
print(f"Průměr studytime - nezadaní: {nezadani.mean():.2f}".replace(".", ","))
print(f"Mann-Whitney U: {u_stat:.2f}".replace(".", ","))
print(f"Velikost účinku (r_rb): {r_rb:.3f}".replace(".", ","))
zaver(p_hodnota)

=== H2: Romantický vztah vs. Čas na studium ===
Průměr studytime - zadaní: 1,97
Průměr studytime - nezadaní: 1,91
Mann-Whitney U: 51899,50
Velikost účinku (r_rb): 0,059
P-hodnota: 0,1735
ROZHODNUTÍ: nezamítáme H0 (α = 0,0071)


**Proč tento test:** `studytime` je ordinální proměnná (1–4), proto místo t-testu používáme Mannův–Whitneyův test, který porovnává pořadí dvou nezávislých skupin a nepředpokládá normalitu ani intervalovou škálu. Test je oboustranný, protože vztah může čas na studium jak ubírat, tak motivovat. Velikost účinku uvádíme pořadovým biseriálním r.

**Co jsme zjistili:** p = 0,174, H0 nezamítáme (ani na hladině 0,05). Průměrný čas samostudia je u zadaných 1,97 a u nezadaných 1,91, velikost účinku je zanedbatelná (r ≈ 0,06). Data tedy neprokázala, že by se čas věnovaný samostudiu lišil podle toho, zda je student v romantickém vztahu. Nezamítnutí neznamená důkaz, že rozdíl neexistuje, jen že se v datech neprojevil.

In [28]:
# === H3: VŠ rodiče vs. Známka G3 ===
vs_rodice = df[df['max_edu'] == 4]['G3']
bez_vs = df[df['max_edu'] < 4]['G3']
t_stat, p_hodnota = stats.ttest_ind(vs_rodice, bez_vs, equal_var=False, alternative='greater')

n1, n2 = len(vs_rodice), len(bez_vs)
s1, s2 = vs_rodice.std(ddof=1), bez_vs.std(ddof=1)
s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
cohen_d = (vs_rodice.mean() - bez_vs.mean()) / s_pooled

print("=== H3: VŠ rodiče vs. Známka G3 ===")
print(f"Průměr G3 - VŠ rodiče: {vs_rodice.mean():.2f}".replace(".", ","))
print(f"Průměr G3 - bez VŠ: {bez_vs.mean():.2f}".replace(".", ","))
print(f"T-statistika: {t_stat:.3f}".replace(".", ","))
print(f"Cohenovo d: {cohen_d:.3f}".replace(".", ","))
zaver(p_hodnota)

# Doplňující kontrola vlivu školy na distribuci VŠ rodičů
podil_skola_0 = (df[df['school'] == 0]['max_edu'] == 4).mean() * 100
podil_skola_1 = (df[df['school'] == 1]['max_edu'] == 4).mean() * 100
print(f"\nDoplňující info: Podíl studentů s VŠ rodiči ve škole 0 je {podil_skola_0:.1f} % a ve škole 1 je {podil_skola_1:.1f} %.".replace(".", ","))

=== H3: VŠ rodiče vs. Známka G3 ===
Průměr G3 - VŠ rodiče: 12,96
Průměr G3 - bez VŠ: 11,41
T-statistika: 5,787
Cohenovo d: 0,491
P-hodnota: < 0,001
ROZHODNUTÍ: zamítáme H0 (α = 0,0071)

Doplňující info: Podíl studentů s VŠ rodiči ve škole 0 je 38,8 % a ve škole 1 je 19,5 %,


**Proč tento test:** `G3` (0–20) je téměř spojitá a skupiny jsou dostatečně velké (208 vs. 441 studentů), takže t-test je robustní i při mírné nenormalitě. Používáme Welchovu variantu, protože nepředpokládá stejné rozptyly a je bezpečnější při nestejných velikostech skupin. Test je jednostranný, protože hypotéza předpokládá vyšší skóre u studentů s VŠ rodičem. Velikost účinku uvádíme Cohenovým d.

**Co jsme zjistili:** Studenti, kteří mají alespoň jednoho rodiče s VŠ, mají průměrné `G3` 12,96, ostatní 11,41 (rozdíl 1,55 bodu). Test vychází p < 0,001, takže H0 zamítáme. Cohenovo d = 0,49 je prakticky střední účinek (hranice je 0,5). Je třeba mít na paměti, že VŠ rodiče jsou častější ve škole 0 (39 % studentů) než ve škole 1 (19 %) a školy se liší průměrnou známkou. Část rozdílu proto může jít na vrub školy. Jde o souvislost, ne o příčinu.

In [29]:
chteji_vs = df[df['higher'] == 1]['max_edu']
nechteji_vs = df[df['higher'] == 0]['max_edu']
u_stat, p_hodnota = stats.mannwhitneyu(chteji_vs, nechteji_vs, alternative='greater')
r_rb = (2 * u_stat) / (len(chteji_vs) * len(nechteji_vs)) - 1

print("=== H4: Vzdělání rodičů vs. Zájem o VŠ ===")
print(f"Průměrné vzdělání rodičů - chtějí na VŠ: {chteji_vs.mean():.2f}".replace(".", ","))
print(f"Průměrné vzdělání rodičů - nechtějí na VŠ: {nechteji_vs.mean():.2f}".replace(".", ","))
print(f"Mann-Whitney U: {u_stat:.2f}".replace(".", ","))
print(f"Velikost účinku (r_rb): {r_rb:.3f}".replace(".", ","))
zaver(p_hodnota)

=== H4: Vzdělání rodičů vs. Zájem o VŠ ===
Průměrné vzdělání rodičů - chtějí na VŠ: 2,80
Průměrné vzdělání rodičů - nechtějí na VŠ: 2,03
Mann-Whitney U: 28044,50
Velikost účinku (r_rb): 0,402
P-hodnota: < 0,001
ROZHODNUTÍ: zamítáme H0 (α = 0,0071)


**Proč tento test:** `max_edu` je ordinální proměnná (0–4) a skupiny jsou velmi nestejné (580 studentů, kteří chtějí na VŠ, vs. 69, kteří nechtějí) s různými rozptyly (Levenův test p < 0,001). Proto používáme neparametrický Mannův–Whitneyův test, který nepředpokládá normalitu ani stejné rozptyly. Test je jednostranný, protože hypotéza předpokládá, že rodiče studentů se zájmem o VŠ mají vyšší vzdělání.

**Co jsme zjistili:** p < 0,001, H0 zamítáme. Studenti, kteří chtějí na VŠ, mají rodiče s vyšším nejvyšším dosaženým vzděláním (průměrné skóre 2,80 vs. 2,03). Velikost účinku (pořadové biseriální r ≈ 0,40) je střední. Skupina studentů, kteří na VŠ nechtějí, je malá (69), takže odhady v ní jsou méně přesné. Jde o souvislost, ne o příčinu.

In [20]:
# === H5: Kvalita rodinných vztahů vs. Alkohol ===
r_stat, p_hodnota = stats.spearmanr(df['famrel'], df['alkohol_celkem'], alternative='less')

print("\n=== H5: Kvalita rodinných vztahů vs. Alkohol ===")
print(f"Spearmanovo rho (r): {r_stat:.3f}".replace(".", ","))
zaver(p_hodnota)


=== H5: Kvalita rodinných vztahů vs. Alkohol ===
Spearmanovo rho (r): -0,113
P-hodnota: 0,0019
ROZHODNUTÍ: zamítáme H0 (α = 0,0071)


**Proč tento test:** `famrel` (1–5) i součet `Dalc + Walc` jsou ordinální škály, ne měření skutečného množství alkoholu, proto používáme Spearmanovu korelaci. Součet je jen souhrnné skóre konzumace. Hypotéza předpokládá záporný vztah, takže test je jednostranný.

**Co jsme zjistili:** ρ = −0,113, p = 0,0019, tedy pod hranicí 0,007, a H0 zamítáme. Studenti s lepšími rodinnými vztahy mají v průměru mírně nižší skóre konzumace alkoholu. Vztah je ale velmi slabý (ρ² ≈ 1 %) a významnost vychází hlavně z velkého počtu studentů (n = 649). Nelze z toho usuzovat, že rodinné vztahy konzumaci snižují.

In [21]:
# === H6: Alkohol vs. Známka G3 ===
r_stat, p_hodnota = stats.spearmanr(df['alkohol_celkem'], df['G3'], alternative='less')

print("\n=== H6: Alkohol vs. Známka (G3) ===")
print(f"Spearmanovo rho (r): {r_stat:.3f}".replace(".", ","))
zaver(p_hodnota)


=== H6: Alkohol vs. Známka (G3) ===
Spearmanovo rho (r): -0,195
P-hodnota: < 0,001
ROZHODNUTÍ: zamítáme H0 (α = 0,0071)


**Proč tento test:** Stejně jako u H5 jde o ordinální proměnné, proto Spearmanova korelace. Hypotéza předpokládá záporný vztah, test je jednostranný.

**Co jsme zjistili:** ρ = −0,195, p < 0,001, H0 zamítáme. Vyšší skóre konzumace alkoholu je spojeno s nižším `G3`. Vztah je slabý až mírný (ρ² ≈ 4 %), takže alkohol je jen jedním z mnoha faktorů, které se se známkami pojí. Jde o souvislost, ne o příčinu.

In [22]:
# === H7: Město vs. Vesnice (po zohlednění školy) ===
model = smf.ols('G3 ~ address + school', data=df).fit()
p_hodnota = model.pvalues['address']

print("\n=== H7: Město vs. Vesnice (po zohlednění školy) ===")
print(model.summary().tables[1])
print(f"\nKoeficient pro adresu (address): {model.params['address']:.3f}".replace(".", ","))
zaver(p_hodnota)


=== H7: Město vs. Vesnice (po zohlednění školy) ===
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     12.1390      0.275     44.122      0.000      11.599      12.679
address        0.5368      0.282      1.901      0.058      -0.018       1.091
school        -1.7427      0.273     -6.393      0.000      -2.278      -1.207

Koeficient pro adresu (address): 0,537
P-hodnota: 0,0578
ROZHODNUTÍ: nezamítáme H0 (α = 0,0071)


**Proč tento test:** Chceme porovnat město a vesnici tak, aby výsledek nezkreslil rozdíl mezi školami. Školy se liší průměrem `G3` (12,58 vs. 10,65) i podílem městských studentů (82 % vs. 47 %). Lineární regrese `G3 ~ address + school` odhadne rozdíl město–vesnice při srovnání studentů stejné školy. Test koeficientu je oboustranný.

**Co jsme zjistili:** P-hodnota pro vliv adresy (0,058) leží nad hladinou významnosti, nulovou hypotézu proto nezamítáme. Po zohlednění školy mají městští studenti v průměru o 0,54 bodu vyšší finální skóre (G3), přičemž 95% interval spolehlivosti leží mezi −0,02 až 1,09. Bez zohlednění vlivu školy byl tento rozdíl 1,17 bodu a jevil se jako velmi významný. Po zahrnutí školy do modelu se však rozdíl zhruba zpolovil a statistická významnost zmizela. Velká část původního rozdílu se tedy dá vysvětlit tím, že městští studenti častěji chodí do školy s celkově vyššími známkami (samotná škola má na skóre výrazný vliv: −1,74 bodu, p < 0,001). Nezamítnutí nulové hypotézy nutně neznamená, že vliv adresy neexistuje, pouze to, že jej tato data po očištění o vliv školy spolehlivě neprokázala.

## Souhrn výsledků

Všech 7 hypotéz jsme testovali na hladině α = 0,05 / 7 ≈ 0,007 (Bonferroniho korekce).
Potvrdilo se **5 ze 7** hypotéz.

| H | Hypotéza | Test | Výsledek | Velikost účinku | Rozhodnutí |
|---|----------|------|----------|-----------------|------------|
| 1 | Více samostudia → vyšší G3 | Spearman (jednostranný) | p < 0,001 | ρ = 0,275 (slabý až mírný) | potvrzeno |
| 2 | Romantický vztah × čas na samostudium | Mannův–Whitneyův | p = 0,174 | r = 0,06 (zanedbatelný) | nepotvrzeno |
| 3 | VŠ rodič → vyšší G3 | Welchův t-test (jednostranný) | p < 0,001 | d = 0,49 (střední) | potvrzeno |
| 4 | Vzdělání rodičů × zájem o VŠ | Mannův–Whitneyův (jednostranný) | p < 0,001 | r = 0,40 (střední) | potvrzeno |
| 5 | Lepší rodinné vztahy → nižší alkohol | Spearman (jednostranný) | p = 0,0019 | ρ = −0,113 (velmi slabý) | potvrzeno |
| 6 | Více alkoholu → nižší G3 | Spearman (jednostranný) | p < 0,001 | ρ = −0,195 (slabý až mírný) | potvrzeno |
| 7 | Město × vesnice po zohlednění školy | Lineární regrese | p = 0,058 | +0,54 bodu | nepotvrzeno |

### Hlavní zjištění

- **Rodinné zázemí a školní chování souvisí se známkami.** Studenti, kteří věnují více času samostudiu, mají v průměru vyšší G3 (H1). Studenti s alespoň jedním rodičem s VŠ mají G3 vyšší o 1,55 bodu (H3) a častěji chtějí sami studovat na VŠ (H4). Vyšší konzumace alkoholu je spojena s nižším G3 (H6).
- **Nejsilnější vztahy** jsou u vzdělání rodičů (H3, H4). Ostatní potvrzené vztahy jsou slabé a při velkém počtu studentů (n = 649) vychází významně i drobné rozdíly.
- **Romantický vztah** se s časem na samostudium v datech nesouvisel (H2).
- **Rozdíl mezi městem a vesnicí** (H7) z velké části vysvětluje škola. Městští studenti častěji chodí do školy s vyššími známkami. Po zohlednění školy se rozdíl zhruba zpolovinil na 0,54 bodu a přestal být významný.

### Omezení

- Jde o **souvislosti, ne o příčiny**. Z dat nelze určit, co co ovlivňuje.
- Data pocházejí jen ze **dvou škol** a mnoho proměnných jsou **subjektivní škály** (samostudium, alkohol, rodinné vztahy). Součet `Dalc + Walc` je souhrnné skóre, ne skutečné množství alkoholu.
- U H3 jsme vliv školy zvlášť neošetřili. VŠ rodiče jsou častější ve škole 0 (39 %) než ve škole 1 (19,5 %), takže část rozdílu může jít na vrub školy.
- Bonferroniho korekce je přísná. Slabé reálné efekty (např. H2 nebo H7) mohly zůstat neprokázané.